In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# Load the cleaned modeling dataset
data_path = Path("../data/processed/rossmann_modeling.csv")

df = pd.read_csv(
    data_path,
    parse_dates=["Date"],
    low_memory=False
)

print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

display(df.head())


Dataset shape: (844338, 23)
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,CompetitionDistanceMissing,Year,Month,Day,WeekOfYear
0,85,2,2013-01-01,4220,619,1,0,a,1,b,...,2011.0,0,NaN,NaN,NaN,0,2013,1,1,1
1,259,2,2013-01-01,6851,1444,1,0,a,1,b,...,NaN,0,NaN,NaN,NaN,0,2013,1,1,1
2,262,2,2013-01-01,17267,2875,1,0,a,1,b,...,2013.0,0,NaN,NaN,NaN,0,2013,1,1,1
3,274,2,2013-01-01,3102,729,1,0,a,1,b,...,NaN,1,10.0,2013.0,"Jan,Apr,Jul,Oct",0,2013,1,1,1
4,335,2,2013-01-01,2401,482,1,0,a,1,b,...,NaN,1,31.0,2013.0,"Jan,Apr,Jul,Oct",0,2013,1,1,1


In [2]:
# Create a copy of the dataset
model_data = df.copy()

# Extract date information
model_data["Month"] = model_data["Date"].dt.month
model_data["Day"] = model_data["Date"].dt.day

# Select input features
features = [
    "Store",
    "DayOfWeek",
    "Month",
    "Day",
    "Promo",
    "SchoolHoliday",
    "StoreType"
]

# Target variable
target = "Sales"

# Display selected features
print("Selected features:", features)
print("Target:", target)

display(model_data[features + [target]].head())


Selected features: ['Store', 'DayOfWeek', 'Month', 'Day', 'Promo', 'SchoolHoliday', 'StoreType']
Target: Sales


,Store,DayOfWeek,Month,Day,Promo,SchoolHoliday,StoreType,Sales
0,85,2,1,1,0,1,b,4220
1,259,2,1,1,0,1,b,6851
2,262,2,1,1,0,1,b,17267
3,274,2,1,1,0,1,b,3102
4,335,2,1,1,0,1,b,2401


In [3]:
# Convert StoreType into numerical columns
model_data = pd.get_dummies(
    model_data,
    columns=["StoreType"],
    dtype=int
)

# Update our feature list
features = [
    "Store",
    "DayOfWeek",
    "Month",
    "Day",
    "Promo",
    "SchoolHoliday",
    "StoreType_a",
    "StoreType_b",
    "StoreType_c",
    "StoreType_d"
]

# Split chronologically (same dates as Week 5)
train_data = model_data[model_data["Date"] < "2015-06-01"]
test_data = model_data[model_data["Date"] >= "2015-06-01"]

# Prepare inputs (X) and target (y)
X_train = train_data[features]
y_train = train_data["Sales"]

X_test = test_data[features]
y_test = test_data["Sales"]

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Number of features:", len(features))

display(X_train.head())


Training records: 785727
Testing records: 58611
Number of features: 10


,Store,DayOfWeek,Month,Day,Promo,SchoolHoliday,StoreType_a,StoreType_b,StoreType_c,StoreType_d
0,85,2,1,1,0,1,0,1,0,0
1,259,2,1,1,0,1,0,1,0,0
2,262,2,1,1,0,1,0,1,0,0
3,274,2,1,1,0,1,0,1,0,0
4,335,2,1,1,0,1,0,1,0,0


In [4]:
from sklearn.tree import DecisionTreeRegressor

# Create the Decision Tree model
decision_tree = DecisionTreeRegressor(
    max_depth=10,
    min_samples_leaf=20,
    random_state=42
)

# Train the model
decision_tree.fit(X_train, y_train)

print("Decision Tree trained successfully!")
print("Tree depth:", decision_tree.get_depth())
print("Number of leaves:", decision_tree.get_n_leaves())


Decision Tree trained successfully!
Tree depth: 10
Number of leaves: 612


In [5]:
# Generate predictions using the Decision Tree
tree_predictions = decision_tree.predict(X_test)

# Compare actual and predicted sales
results = test_data[["Date", "Store", "Sales"]].copy()

results["PredictedSales"] = tree_predictions.round(2)

# Display the first 10 predictions
display(results.head(10))


,Date,Store,Sales,PredictedSales
785727,2015-06-01,1,5774,9491.98
785728,2015-06-01,2,6629,9504.83
785729,2015-06-01,3,11594,9504.83
785730,2015-06-01,4,13727,9491.98
785731,2015-06-01,5,8396,9504.83
785732,2015-06-01,6,7054,9504.83
785733,2015-06-01,7,14160,9504.83
785734,2015-06-01,8,10971,9504.83
785735,2015-06-01,9,12227,9504.83
785736,2015-06-01,10,7846,9504.83


In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Actual and predicted sales
actual = y_test
predicted = tree_predictions

# Calculate evaluation metrics
tree_mae = mean_absolute_error(actual, predicted)
tree_rmse = np.sqrt(mean_squared_error(actual, predicted))

print("Decision Tree Model Evaluation")
print(f"MAE: {tree_mae:.2f}")
print(f"RMSE: {tree_rmse:.2f}")

# Compare with Week 5 baseline
baseline_mae = 2199.64
baseline_rmse = 3009.76

print("\nBaseline vs Decision Tree")
print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Decision Tree MAE: {tree_mae:.2f}")
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Decision Tree RMSE: {tree_rmse:.2f}")

Decision Tree Model Evaluation
MAE: 1880.80
RMSE: 2622.34

Baseline vs Decision Tree
Baseline MAE: 2199.64
Decision Tree MAE: 1880.80
Baseline RMSE: 3009.76
Decision Tree RMSE: 2622.34
